# StatsBomb Open Data — Dataset Exploration

This notebook documents what data is available in the [StatsBomb Open Data](https://github.com/statsbomb/open-data) repository.
It covers:
- Available competitions and seasons
- Match-level data structure
- Event-level data structure and event types
- Lineup data structure
- 360 freeze frame data (where available)

Run cells top to bottom. No credentials required — all data is free and public.

In [ ]:
from statsbombpy import sb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

---
## 1. Available Competitions

`sb.competitions()` returns every competition + season combination available in the open dataset.

In [ ]:
competitions = sb.competitions()
print(f"Total competition/season combinations: {len(competitions)}")
competitions

In [ ]:
# Columns available on the competitions dataframe
print("Columns:")
for col in competitions.columns:
    print(f"  {col}: {competitions[col].dtype} — example: {competitions[col].iloc[0]}")

In [ ]:
# Unique competitions (grouped, ignoring season)
print("Unique competitions:")
print(competitions[['competition_id', 'competition_name', 'country_name']]
      .drop_duplicates()
      .sort_values('competition_name')
      .to_string(index=False))

---
## 2. Matches

`sb.matches(competition_id, season_id)` returns all matches for a given competition + season.

We'll use **UEFA Euro 2020** (competition_id=55, season_id=43) as our reference throughout this notebook.

In [ ]:
# UEFA Euro 2020
COMPETITION_ID = 55
SEASON_ID = 43

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
print(f"Total matches: {len(matches)}")
matches.head()

In [ ]:
# All columns in the matches dataframe
print("Match-level columns:")
for col in matches.columns:
    sample = matches[col].iloc[0]
    print(f"  {col} ({matches[col].dtype}): {sample}")

---
## 3. Lineups

`sb.lineups(match_id)` returns both teams' starting lineups and substitutes for a given match.

Returns a dict with team names as keys, each value being a DataFrame of players.

In [ ]:
# Use the first match as an example
MATCH_ID = matches.iloc[0]['match_id']
match_label = f"{matches.iloc[0]['home_team']} vs {matches.iloc[0]['away_team']}"
print(f"Examining match: {match_label} (match_id={MATCH_ID})")

lineups = sb.lineups(match_id=MATCH_ID)
print(f"\nTeams in lineup: {list(lineups.keys())}")

In [ ]:
# Show one team's lineup
team_name = list(lineups.keys())[0]
lineup_df = lineups[team_name]
print(f"Lineup columns for {team_name}:")
for col in lineup_df.columns:
    print(f"  {col} ({lineup_df[col].dtype})")

lineup_df[['player_id', 'player_name', 'player_nickname', 'jersey_number', 'country']]

---
## 4. Events

`sb.events(match_id)` returns every on-ball action in a match.
This is the richest dataset — each row is one event (pass, shot, carry, dribble, etc.).

This is the primary dataset for ML modeling.

In [ ]:
events = sb.events(match_id=MATCH_ID)
print(f"Total events in match: {len(events)}")
events.head()

In [ ]:
# All columns in the events dataframe
print(f"Total columns: {len(events.columns)}\n")
print("Event columns:")
for col in sorted(events.columns):
    non_null = events[col].notna().sum()
    pct = round(non_null / len(events) * 100, 1)
    print(f"  {col} ({events[col].dtype}) — {non_null}/{len(events)} non-null ({pct}%)")

In [ ]:
# Event types — what kinds of actions are recorded?
print("Event types and counts:")
print(events['type'].value_counts().to_string())

---
## 5. Shots — Key ML Target

Shots are the most commonly modeled event type. Each shot has an `xG` (expected goals) value
and enough context to build a shot quality model from scratch.

Below we examine the shot-specific columns available.

In [ ]:
shots = events[events['type'] == 'Shot'].copy()
print(f"Total shots in match: {len(shots)}")

# Shot-specific columns (non-null on shots)
shot_cols = [col for col in shots.columns if shots[col].notna().sum() > 0 and col.startswith('shot')]
print(f"\nShot-specific columns ({len(shot_cols)}):")
for col in shot_cols:
    print(f"  {col}: {shots[col].dtype}")

shots[['player', 'minute', 'location', 'shot_statsbomb_xg', 'shot_outcome', 'shot_technique', 'shot_body_part', 'shot_type']].head(10)

In [ ]:
# Shot outcomes — what we'd predict in an xG model
print("Shot outcomes:")
print(shots['shot_outcome'].value_counts().to_string())

---
## 6. Passes

In [ ]:
passes = events[events['type'] == 'Pass'].copy()
print(f"Total passes in match: {len(passes)}")

pass_cols = [col for col in passes.columns if col.startswith('pass') and passes[col].notna().sum() > 0]
print(f"\nPass-specific columns ({len(pass_cols)}):")
for col in pass_cols:
    print(f"  {col}: {passes[col].dtype}")

---
## 7. 360 Data (Freeze Frames)

Some competitions include 360 data — the position of every visible player at the moment of each event.
This is only available for select competitions.

`sb.frames(match_id)` returns the freeze frame data.

In [ ]:
# Check which competitions have 360 data
has_360 = competitions[competitions['match_available_360'].notna()]
print(f"Competitions with 360 data available: {len(has_360)}")
has_360[['competition_name', 'season_name', 'match_available_360']]

In [ ]:
# Load 360 frames for our match (if available)
try:
    frames = sb.frames(match_id=MATCH_ID)
    print(f"Total freeze frames: {len(frames)}")
    print("\nFrame columns:")
    for col in frames.columns:
        print(f"  {col}: {frames[col].dtype}")
    frames.head()
except Exception as e:
    print(f"360 data not available for this match: {e}")

---
## 8. Loading Data Across Multiple Matches

For ML, we'll want events from all matches in a competition, not just one.
Below shows how to load and concatenate all events from a full competition.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

all_events = []
for _, match in matches.iterrows():
    match_events = sb.events(match_id=match['match_id'])
    match_events['match_id'] = match['match_id']
    match_events['home_team'] = match['home_team']
    match_events['away_team'] = match['away_team']
    all_events.append(match_events)

all_events_df = pd.concat(all_events, ignore_index=True)
print(f"Total events across all {len(matches)} matches: {len(all_events_df)}")
print(f"Total shots: {(all_events_df['type'] == 'Shot').sum()}")
print(f"Total passes: {(all_events_df['type'] == 'Pass').sum()}")

---
## Summary

| Dataset | How to load | Key use |
|---|---|---|
| Competitions | `sb.competitions()` | Find available competition/season IDs |
| Matches | `sb.matches(competition_id, season_id)` | Get match IDs + metadata |
| Lineups | `sb.lineups(match_id)` | Player rosters, positions |
| Events | `sb.events(match_id)` | All on-ball actions — primary ML dataset |
| Frames (360) | `sb.frames(match_id)` | Player positions at moment of each event |

### ML modeling candidates (using event data)
- **Shot outcome prediction (xG model)** — predict goal vs no-goal from shot features
- **Pass completion prediction** — predict whether a pass is completed
- **Press outcome** — predict whether a press recovers the ball
- **Dribble success** — predict whether a take-on beats the defender